# Entrega Intermedia 2 – Proyecto de Clustering
**MP6: Proyecto de IA y Big Data | Tema 3**

Proyecto end-to-end de segmentación de usuarios sobre el dataset de e-commerce (Online Retail UCI).

| Paso | Contenido |
|------|-----------|
| 1 | Carga y análisis exploratorio |
| 2 | Limpieza de datos (7 pasos justificados) |
| 3 | Feature engineering → dataset centrado en usuarios |
| 4 | Limpieza del dataset de usuarios (capping + Isolation Forest) |
| 5 | Escalado, codificación y PCA |
| 6 | Determinación del K óptimo |
| 7 | Entrenamiento de modelos (K-Means, Agglomerative, DBSCAN) |
| 8 | Evaluación y visualización |
| 9 | Interpretación de clusters |

---
## 0. Imports y configuración

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
RANDOM_STATE = 42

# ─── Ajusta esta ruta al archivo raw ─────────────────────────────────────────
RUTA_CSV = '../../../data/raw/data.csv'
# ─────────────────────────────────────────────────────────────────────────────
print('[OK] Librerías cargadas')

---
## 1. Carga y análisis exploratorio

In [ ]:
df_raw = pd.read_csv(RUTA_CSV, encoding='latin-1')
df = df_raw.copy()

df['InvoiceDate']   = pd.to_datetime(df['InvoiceDate'], format='mixed')
df['Fecha']         = df['InvoiceDate'].dt.normalize()
df['Mes']           = df['InvoiceDate'].dt.to_period('M')
df['DiaSemana']     = df['InvoiceDate'].dt.day_name()
df['EsCancelacion'] = df['InvoiceNo'].astype(str).str.startswith('C')
df['TotalPrice']    = df['Quantity'] * df['UnitPrice']

print(f'Shape: {df.shape}')
print(f'Período: {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')
df.head()

In [ ]:
# Valores nulos
nulos = df.isnull().sum()
display(pd.DataFrame({'Nulos': nulos, '%': (nulos / len(df) * 100).round(2)})[nulos > 0])
df.describe()

In [ ]:
print(f'Clientes únicos (con ID): {df["CustomerID"].nunique():,}')
print(f'Productos únicos:         {df["StockCode"].nunique():,}')
print(f'Facturas únicas:          {df["InvoiceNo"].nunique():,}')
print(f'Países:                   {df["Country"].nunique():,}')
print(f'Cancelaciones:            {df["EsCancelacion"].sum():,} ({df["EsCancelacion"].mean()*100:.1f}%)')

df['Country'].value_counts().head(10).plot(
    kind='bar', title='Top 10 países por nº de transacciones', color='steelblue'
)
plt.ylabel('Transacciones'); plt.tight_layout(); plt.show()

**Observaciones del EDA:**
- `CustomerID` tiene ~25% de nulos → eliminación obligatoria para clustering de usuarios.
- Existen `Quantity < 0` sin prefijo 'C' (ajustes de almacén) y códigos de producto no estándar.
- Las cancelaciones (~16%) se **conservan** como dimensión de comportamiento de usuario.
- Dominio mayoritariamente UK; países con <1% se agruparán en 'Otros'.

---
## 2. Limpieza de datos

| Paso | Acción | Justificación |
|------|--------|---------------|
| 2.1 | Eliminar duplicados exactos | Errores de doble inserción en BD / ETL |
| 2.2 | Eliminar negativos huérfanos | Ajustes internos de almacén sin cliente ni valor |
| 2.3 | Eliminar StockCodes no estándar | Códigos operacionales, no productos reales |
| 2.4 | Eliminar Quantity = 0 | No representan ninguna transacción real |
| 2.5 | Eliminar UnitPrice = 0 | Sin valor económico para el clustering |
| 2.6 | **Eliminar CustomerID nulos** | Obligatorio: sin cliente no existe cluster |
| 2.7 | Eliminar Description nulas | Sin descripción se pierde la dimensión 'qué compra' |
| 2.8 | **Conservar cancelaciones** | Tasa de devolución = patrón válido de comportamiento |
| 2.9 | Capping al p99 | Reduce distorsión en distancias sin eliminar usuarios |

In [ ]:
stats = {'Inicial': len(df)}

# 2.1 Duplicados
antes = len(df); df = df.drop_duplicates(keep='first', ignore_index=True)
stats['Duplicados eliminados'] = antes - len(df)
print(f'2.1 Duplicados eliminados        : {stats["Duplicados eliminados"]:,}')

# 2.2 Negativos huérfanos
antes = len(df)
mask = (~df['InvoiceNo'].astype(str).str.startswith('C', na=False) &
        (df['Quantity'] < 0) & (df['UnitPrice'] == 0.0))
df = df[~mask].reset_index(drop=True)
stats['Negativos huérfanos'] = antes - len(df)
print(f'2.2 Negativos huérfanos          : {stats["Negativos huérfanos"]:,}')

# 2.3 StockCodes no estándar
PATRON_STOCK = r'^[0-9]{5}[A-Za-z]{0,2}$'
antes = len(df)
df = df[df['StockCode'].str.match(PATRON_STOCK, na=False)].reset_index(drop=True)
stats['StockCodes no estándar'] = antes - len(df)
print(f'2.3 StockCodes no estándar       : {stats["StockCodes no estándar"]:,}')

# 2.4 Quantity = 0
antes = len(df); df = df[df['Quantity'] != 0].reset_index(drop=True)
stats['Quantity = 0'] = antes - len(df)
print(f'2.4 Quantity = 0                 : {stats["Quantity = 0"]:,}')

# 2.5 UnitPrice = 0
antes = len(df); df = df[df['UnitPrice'] > 0].reset_index(drop=True)
stats['UnitPrice = 0'] = antes - len(df)
print(f'2.5 UnitPrice = 0                : {stats["UnitPrice = 0"]:,}')

# 2.6 CustomerID nulos (OBLIGATORIO)
antes = len(df); df = df.dropna(subset=['CustomerID']).reset_index(drop=True)
stats['CustomerID nulos'] = antes - len(df)
print(f'2.6 CustomerID nulos (OBLIGATORIO): {stats["CustomerID nulos"]:,}')

# 2.7 Description nulas
antes = len(df); df = df.dropna(subset=['Description']).reset_index(drop=True)
stats['Description nulas'] = antes - len(df)
print(f'2.7 Description nulas            : {stats["Description nulas"]:,}')

# 2.8 Cancelaciones: CONSERVAR
stats['Cancelaciones conservadas'] = df['EsCancelacion'].sum()
print(f'2.8 Cancelaciones conservadas    : {stats["Cancelaciones conservadas"]:,}')

df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print(f'\nDataset limpio: {df.shape[0]:,} filas | {df["CustomerID"].nunique():,} usuarios')

In [ ]:
# 2.9 Capping al p99
p99_qty   = df.loc[df['Quantity'] > 0, 'Quantity'].quantile(0.99)
p99_price = df['UnitPrice'].quantile(0.99)
df.loc[df['Quantity'] > p99_qty, 'Quantity'] = p99_qty
df.loc[df['UnitPrice'] > p99_price, 'UnitPrice'] = p99_price
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
print(f'P99 Quantity: {p99_qty:,.0f} | P99 UnitPrice: £{p99_price:,.2f}')

# Balance visual
elim = stats['Inicial'] - len(df)
plt.figure(figsize=(5, 4))
plt.pie([len(df), elim],
        labels=[f'Conservadas\n{len(df):,}', f'Eliminadas\n{elim:,}'],
        colors=['steelblue', 'salmon'], autopct='%1.1f%%', startangle=90)
plt.title('Balance de la limpieza')
plt.tight_layout(); plt.show()

---
## 3. Feature Engineering – Dataset centrado en usuarios

Construimos 20 variables por usuario en 6 grupos.

In [ ]:
df_compras     = df[~df['EsCancelacion']].copy()
df_cancelacion = df[df['EsCancelacion']].copy()
fecha_ref      = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# ── RFM ──────────────────────────────────────────────────────────────────────
recency   = (fecha_ref - df_compras.groupby('CustomerID')['InvoiceDate'].max()).dt.days
frequency = df_compras.groupby('CustomerID')['InvoiceNo'].nunique()
monetary  = df_compras.groupby('CustomerID')['TotalPrice'].sum()
df_rfm    = pd.DataFrame({'Recency': recency, 'Frequency': frequency, 'Monetary': monetary}).reset_index()
print(f'[OK] RFM: {df_rfm.shape[0]:,} usuarios')

In [ ]:
# ── Frecuencia temporal ───────────────────────────────────────────────────────
df_compras['anio_mes'] = df_compras['InvoiceDate'].dt.to_period('M')
num_meses    = df_compras.groupby('CustomerID')['anio_mes'].nunique()
primera      = df_compras.groupby('CustomerID')['InvoiceDate'].min()
ultima       = df_compras.groupby('CustomerID')['InvoiceDate'].max()
periodo      = (ultima - primera).dt.days

df_rfm = df_rfm.merge(num_meses.rename('num_meses_activo'), on='CustomerID', how='left')
df_rfm['num_meses_activo'] = df_rfm['num_meses_activo'].fillna(1)
df_rfm['frecuencia_mensual'] = np.where(
    df_rfm['num_meses_activo'] == 0, 0,
    (df_rfm['Frequency'] / df_rfm['num_meses_activo']).round(2))
df_rfm = df_rfm.merge(periodo.rename('periodo_activo'), on='CustomerID', how='left')
df_rfm['dias_entre_compras'] = np.where(
    df_rfm['Frequency'] <= 1, 0,
    (df_rfm['periodo_activo'] / (df_rfm['Frequency'] - 1)).round(1))
df_rfm = df_rfm.drop(columns='periodo_activo')
print('[OK] Frecuencia temporal')

In [ ]:
# ── Valor de compra ───────────────────────────────────────────────────────────
df_rfm['ticket_promedio'] = np.where(df_rfm['Frequency'] == 0, 0,
    (df_rfm['Monetary'] / df_rfm['Frequency']).round(2))
df_rfm['gasto_mensual']   = np.where(df_rfm['num_meses_activo'] == 0, 0,
    (df_rfm['Monetary'] / df_rfm['num_meses_activo']).round(2))

# ── Diversidad de productos ───────────────────────────────────────────────────
n_prod = df_compras.groupby('CustomerID')['StockCode'].nunique()
df_rfm = df_rfm.merge(n_prod.rename('num_productos_unicos'), on='CustomerID', how='left')
df_rfm['num_productos_unicos'] = df_rfm['num_productos_unicos'].fillna(0).astype(int)
df_rfm['diversidad_producto']  = np.where(df_rfm['Frequency'] == 0, 0,
    (df_rfm['num_productos_unicos'] / df_rfm['Frequency']).round(2))

# ── Cantidad de productos ─────────────────────────────────────────────────────
cant_total = df_compras[df_compras['Quantity'] > 0].groupby('CustomerID')['Quantity'].sum()
df_rfm = df_rfm.merge(cant_total.rename('cantidad_total_comprada'), on='CustomerID', how='left')
df_rfm['cantidad_total_comprada']    = df_rfm['cantidad_total_comprada'].fillna(0).astype(int)
df_rfm['cantidad_promedio_por_compra'] = np.where(df_rfm['Frequency'] == 0, 0,
    (df_rfm['cantidad_total_comprada'] / df_rfm['Frequency']).round(1))
print('[OK] Valor, diversidad y cantidad')

In [ ]:
# ── Variables de devolución ───────────────────────────────────────────────────
n_cancel   = df_cancelacion.groupby('CustomerID')['InvoiceNo'].nunique()
v_devuelto = df_cancelacion.groupby('CustomerID')['TotalPrice'].sum().abs()

df_rfm = df_rfm.merge(n_cancel.rename('num_cancelaciones'), on='CustomerID', how='left')
df_rfm['num_cancelaciones'] = df_rfm['num_cancelaciones'].fillna(0).astype(int)
df_rfm['tasa_cancelacion']  = np.where(df_rfm['Frequency'] == 0, 0,
    (df_rfm['num_cancelaciones'] / df_rfm['Frequency']).round(4))
df_rfm = df_rfm.merge(v_devuelto.rename('valor_devuelto'), on='CustomerID', how='left')
df_rfm['valor_devuelto'] = df_rfm['valor_devuelto'].fillna(0).round(2)
df_rfm['ratio_devolucion_monetario'] = np.where(df_rfm['Monetary'] == 0, 0,
    (df_rfm['valor_devuelto'] / df_rfm['Monetary']).round(4))

# ── Comportamiento temporal semanal ───────────────────────────────────────────
dia_modo = df_compras.groupby('CustomerID')['DiaSemana'].agg(lambda x: x.value_counts().index[0])
compras_finde = (df_compras[df_compras['DiaSemana'].isin({'Saturday','Sunday'})]
                 .groupby('CustomerID')['InvoiceNo'].nunique())
df_rfm = df_rfm.merge(dia_modo.rename('dia_semana_mas_frecuente'), on='CustomerID', how='left')
df_rfm['dia_semana_mas_frecuente'] = df_rfm['dia_semana_mas_frecuente'].fillna('Unknown')
df_rfm = df_rfm.merge(compras_finde.rename('compras_fin_semana'), on='CustomerID', how='left')
df_rfm['compras_fin_semana'] = df_rfm['compras_fin_semana'].fillna(0).astype(int)
df_rfm['ratio_fin_semana']   = np.where(df_rfm['Frequency'] == 0, 0,
    (df_rfm['compras_fin_semana'] / df_rfm['Frequency']).round(4))

# ── Variable geográfica ───────────────────────────────────────────────────────
pais_s = df.groupby('CustomerID')['Country'].agg(lambda x: x.value_counts().index[0])
n_total = pais_s.shape[0]
paises_ok = set(pais_s.value_counts()[pais_s.value_counts() / n_total >= 0.01].index)
pais_s = pais_s.apply(lambda p: p if p in paises_ok else 'Otros')
df_rfm = df_rfm.merge(pais_s.rename('pais'), on='CustomerID', how='left')
df_rfm['pais'] = df_rfm['pais'].fillna('Otros')

print(f'Dataset de usuarios: {df_rfm.shape}')
display(df_rfm.head())

In [ ]:
# Distribuciones RFM
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col, color in zip(axes,
    ['Recency', 'Frequency', 'Monetary'],
    ['steelblue', 'seagreen', 'darkorange']):
    ax.hist(df_rfm[col], bins=50, color=color, edgecolor='black', alpha=0.75)
    ax.axvline(df_rfm[col].mean(),   color='red',   linestyle='--',
               label=f'Media: {df_rfm[col].mean():.1f}')
    ax.axvline(df_rfm[col].median(), color='black', linestyle=':',
               label=f'Mediana: {df_rfm[col].median():.1f}')
    ax.set_title(col, fontweight='bold'); ax.set_yscale('log'); ax.legend(fontsize=9)
plt.suptitle('Distribución RFM (escala log)', fontsize=13)
plt.tight_layout(); plt.show()

---
## 4. Limpieza del dataset de usuarios
### 4.1 Capping suave al p99 sobre features numéricas del dataset de usuarios

In [ ]:
COLS_NUMERICAS = [
    'Recency', 'Frequency', 'Monetary',
    'num_meses_activo', 'frecuencia_mensual', 'dias_entre_compras',
    'ticket_promedio', 'gasto_mensual',
    'num_productos_unicos', 'diversidad_producto',
    'cantidad_total_comprada', 'cantidad_promedio_por_compra',
    'num_cancelaciones', 'tasa_cancelacion', 'valor_devuelto',
    'ratio_devolucion_monetario', 'compras_fin_semana', 'ratio_fin_semana',
]
COLS_CATEGORICAS = ['dia_semana_mas_frecuente', 'pais']
FEATURES_CAPPING = [
    'Frequency', 'Monetary', 'num_meses_activo', 'frecuencia_mensual',
    'ticket_promedio', 'gasto_mensual', 'num_productos_unicos',
    'cantidad_total_comprada', 'cantidad_promedio_por_compra',
    'num_cancelaciones', 'valor_devuelto'
]
df_users = df_rfm.copy()
for col in FEATURES_CAPPING:
    p99 = df_users[col].quantile(0.99)
    df_users.loc[df_users[col] > p99, col] = p99
print(f'[OK] Capping p99 aplicado a {len(FEATURES_CAPPING)} features')

### 4.2 Isolation Forest – Outliers multivariantes

**Justificación**: El capping elimina extremos univariantes, pero pueden existir combinaciones anómalas de varias features. `IsolationForest` con `contamination=0.05` detecta el 5% más anómalo, dentro del rango recomendado (5-10%).

In [ ]:
scaler_if = RobustScaler()
X_for_if  = scaler_if.fit_transform(df_users[COLS_NUMERICAS])

iso      = IsolationForest(contamination=0.05, n_estimators=200, random_state=RANDOM_STATE)
lbl_if   = iso.fit_predict(X_for_if)
n_out    = (lbl_if == -1).sum()
print(f'Outliers detectados: {n_out} ({n_out/len(df_users)*100:.1f}%)')

# Visualización en PCA 2D
pca_vis = PCA(n_components=2, random_state=RANDOM_STATE)
X_vis   = pca_vis.fit_transform(X_for_if)
plt.figure(figsize=(9, 5))
plt.scatter(X_vis[:, 0], X_vis[:, 1],
            c=['red' if l == -1 else 'steelblue' for l in lbl_if],
            alpha=0.5, s=12)
plt.title('Isolation Forest – Outliers detectados (rojo)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(handles=[
    mpatches.Patch(color='steelblue', label=f'Normal ({(lbl_if==1).sum():,})'),
    mpatches.Patch(color='red',       label=f'Outlier ({n_out:,})')
])
plt.tight_layout(); plt.show()

df_users = df_users[lbl_if == 1].copy().reset_index(drop=True)
print(f'Dataset final: {df_users.shape[0]:,} usuarios')

---
## 5. Escalado, codificación y PCA

In [ ]:
# 5.1 One-Hot encoding de variables categóricas
df_enc = pd.get_dummies(df_users, columns=COLS_CATEGORICAS, drop_first=False)
feature_cols = [c for c in df_enc.columns if c != 'CustomerID']
X = df_enc[feature_cols].copy()
print(f'Features tras encoding: {X.shape[1]}')

In [ ]:
# 5.2 RobustScaler
# Justificación: usa mediana e IQR → resistente a outliers residuales
# Los algoritmos de clustering usan distancias → escala homogénea imprescindible
scaler  = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

# Heatmap de correlación
corr = X_scaled[COLS_NUMERICAS].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.1f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlación entre features numéricas escaladas')
plt.tight_layout(); plt.show()
print('Alta correlación justifica reducción de dimensionalidad con PCA')

In [ ]:
# 5.3 PCA – varianza explicada acumulada
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(var_acum)+1), var_acum * 100, marker='o', markersize=4)
plt.axhline(80, color='red',    linestyle='--', label='80%')
plt.axhline(90, color='orange', linestyle='--', label='90%')
plt.xlabel('Número de componentes'); plt.ylabel('Varianza explicada acumulada (%)')
plt.title('PCA – Varianza explicada acumulada'); plt.legend()
plt.tight_layout(); plt.show()

n_components = int(np.argmax(var_acum >= 0.80)) + 1
print(f'Componentes para ≥80% varianza: {n_components}')
print(f'Varianza explicada: {var_acum[n_components-1]*100:.1f}%')

In [ ]:
pca    = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_pca  = pca.fit_transform(X_scaled)
print(f'Shape tras PCA: {X_pca.shape}')
print(f'Varianza total explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%')

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(1, n_components+1), pca.explained_variance_ratio_*100, color='steelblue')
ax.set_xlabel('Componente principal'); ax.set_ylabel('% Varianza')
ax.set_title('Scree Plot')
plt.tight_layout(); plt.show()

---
## 6. Determinación del K óptimo

In [ ]:
inertias = []; silhouettes = []; K_range = range(2, 11)
for k in K_range:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
    lbl = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, lbl))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias,    marker='o', color='steelblue')
axes[0].set_title('Método del Codo (Elbow)'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[1].plot(K_range, silhouettes, marker='o', color='darkorange')
axes[1].set_title('Índice de Silueta'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
plt.suptitle('Determinación del K óptimo', fontsize=13)
plt.tight_layout(); plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'K con mayor silueta: {best_k}')

In [ ]:
# Fija K aquí (puedes cambiarlo si el codo sugiere otro valor)
K = best_k
print(f'K elegido: {K}')

---
## 7. Entrenamiento de modelos de clustering

In [ ]:
# ── 7.1 K-Means ──────────────────────────────────────────────────────────────
kmeans    = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init='auto')
km_labels = kmeans.fit_predict(X_pca)
sil_km    = silhouette_score(X_pca, km_labels)
db_km     = davies_bouldin_score(X_pca, km_labels)
df_users['cluster_kmeans'] = km_labels
print(f'K-Means (K={K}):  Silhouette={sil_km:.4f}  Davies-Bouldin={db_km:.4f}')

In [ ]:
# ── 7.2 Agglomerative Clustering ─────────────────────────────────────────────
agglo     = AgglomerativeClustering(n_clusters=K)
ag_labels = agglo.fit_predict(X_pca)
sil_ag    = silhouette_score(X_pca, ag_labels)
db_ag     = davies_bouldin_score(X_pca, ag_labels)
df_users['cluster_agglo'] = ag_labels
print(f'Agglomerative (K={K}): Silhouette={sil_ag:.4f}  Davies-Bouldin={db_ag:.4f}')

In [ ]:
# ── 7.3 DBSCAN ───────────────────────────────────────────────────────────────
# Ajusta eps si el resultado no es satisfactorio (prueba 0.5, 1.0, 1.5, 2.0)
dbscan    = DBSCAN(eps=1.5, min_samples=10)
db_labels = dbscan.fit_predict(X_pca)
n_cl_db   = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise   = (db_labels == -1).sum()
print(f'DBSCAN → Clusters: {n_cl_db} | Ruido: {n_noise}')
if n_cl_db > 1:
    mask   = db_labels != -1
    sil_db = silhouette_score(X_pca[mask], db_labels[mask])
    db_db  = davies_bouldin_score(X_pca[mask], db_labels[mask])
    print(f'  Silhouette={sil_db:.4f}  Davies-Bouldin={db_db:.4f}')
else:
    print('  Ajusta eps o min_samples para obtener múltiples clusters')
df_users['cluster_dbscan'] = db_labels

In [ ]:
# Comparativa
display(pd.DataFrame({
    'Modelo': ['K-Means', 'Agglomerative'],
    'Silhouette ↑': [sil_km, sil_ag],
    'Davies-Bouldin ↓': [db_km, db_ag],
}))

---
## 8. Visualización de clusters

In [ ]:
X_2d = X_pca[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, labels, title in zip(axes,
    [km_labels, ag_labels], ['K-Means', 'Agglomerative']):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='tab10', alpha=0.6, s=12)
    ax.set_title(f'{title} (K={K})', fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.colorbar(sc, ax=ax, label='Cluster')
plt.suptitle('Clusters en PCA 2D', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Tamaño de clusters
df_users['cluster_kmeans'].value_counts().sort_index().plot(
    kind='bar', color='steelblue',
    title=f'Nº de usuarios por cluster (K-Means, K={K})')
plt.xlabel('Cluster'); plt.ylabel('Nº usuarios')
plt.tight_layout(); plt.show()

---
## 9. Interpretación de clusters

In [ ]:
key_feat = ['Recency', 'Frequency', 'Monetary', 'ticket_promedio',
            'num_productos_unicos', 'tasa_cancelacion', 'dias_entre_compras']

profile = df_users.groupby('cluster_kmeans')[key_feat].median().round(2)
profile['n_usuarios'] = df_users['cluster_kmeans'].value_counts().sort_index()
profile['%_usuarios'] = (profile['n_usuarios'] / len(df_users) * 100).round(1)
display(profile)

In [ ]:
# Heatmap de perfiles
p_norm = (profile[key_feat] - profile[key_feat].min()) / \
         (profile[key_feat].max() - profile[key_feat].min())
plt.figure(figsize=(12, max(3, K)))
sns.heatmap(p_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white')
plt.title('Perfil normalizado de clusters (K-Means)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Boxplots de variables clave
vars_box = ['Monetary', 'Recency', 'Frequency', 'tasa_cancelacion']
fig, axes = plt.subplots(1, len(vars_box), figsize=(18, 5))
for ax, var in zip(axes, vars_box):
    df_users.boxplot(column=var, by='cluster_kmeans', ax=ax)
    plt.sca(ax); plt.title(var, fontweight='bold')
plt.suptitle('Distribución por cluster', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 9.1 Conclusiones y nombres descriptivos

> **Completa esta tabla con los valores reales** del perfil mediano obtenido arriba.

| Cluster | Recency | Frequency | Monetary | tasa_cancel | Nombre descriptivo |
|---------|---------|-----------|----------|-------------|--------------------|
| 0 | ... | ... | ... | ... | ej. "Clientes VIP activos" |
| 1 | ... | ... | ... | ... | ej. "Clientes en riesgo" |
| 2 | ... | ... | ... | ... | ej. "Clientes regulares" |

**Guía rápida:**
- Recency bajo + Frequency alta + Monetary alto → **VIP activos**
- Recency alto → **Inactivos / en riesgo de pérdida**
- tasa_cancelacion alta → **Compradores problemáticos**
- Frequency = 1 → **Compradores de una sola compra**

**Implicaciones de negocio:**
- VIP activos → programa de fidelización premium
- Inactivos → campaña de reactivación con incentivo
- Alta cancelación → mejorar descripciones de producto / política de devoluciones
- Nuevos → incentivo de segunda compra para convertirlos en regulares